# المشروع الصغير — هل هذه السيارة موفّرة للوقود؟
## برنامج كاوست الصيفي للذكاء الاصطناعي · الأسبوع الثاني · اليوم الخامس

في المعمل قيّمنا نموذج الأزهار **بأيدينا**. اليوم في المشروع نقيّم نموذجًا على بيانات **السيارات**،
ونتعلّم مهارتين معًا: الحساب **بأيدينا**، ثمّ التحقّق منه بمكتبة **scikit-learn**.

**القصّة.** تطبيقٌ لبيع السيارات يريد أن يضع شارةً خضراء «موفّرة للوقود» على السيارات المناسبة.
شارةٌ خاطئةٌ تُضلّل المشتري. مهمّتك أن تبني نموذجًا بسيطًا وتقيّمه: هل نثق به قبل إطلاق الشارة؟

الفئة الموجبة (1) = **سيارة موفّرة** (‏mpg ≥ 25).

### 1) حمّل بيانات السيارات واصنع الهدف الثنائي

In [ ]:
%pip install -q scikit-learn

import pandas as pd

# البيانات تُحمَّل من الإنترنت مباشرة، لا حاجة لرفع أي ملف
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"
try:
    from pyodide.http import open_url
    df = pd.read_csv(open_url(url)).dropna()
except ImportError:
    df = pd.read_csv(url).dropna()

# نحول سؤال «كم ميلًا لكل جالون؟» إلى سؤال نعم/لا:
df["efficient"] = (df["mpg"] >= 25).astype(int)   # 1 = موفرة للوقود

print("cars:", len(df))
print(df["efficient"].value_counts())
df[["name", "weight", "mpg", "efficient"]].head()

### 2) قسّم البيانات، وابنِ نموذجًا بسيطًا
السيارات الأخفّ عادةً أوفر للوقود. القاعدة: نتوقّع «موفّرة» إذا كان الوزن ≤ العتبة.
نبحث عن أفضل عتبة وزنٍ على بيانات **التدريب**.

In [2]:
from sklearn.model_selection import train_test_split

X = df["weight"].tolist()
y = df["efficient"].tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

def predict(weight, threshold):
    # efficient if the car is light enough
    if weight <= threshold:
        return 1
    return 0

best_t, best_acc = None, -1
for t in range(1800, 3600, 50):
    correct = sum(1 for i in range(len(X_train)) if predict(X_train[i], t) == y_train[i])
    acc = correct / len(X_train)
    if acc > best_acc:
        best_acc, best_t = acc, t
print("best weight threshold:", best_t, "lb  (train accuracy", round(best_acc, 3), ")")

best weight threshold: 2700 lb  (train accuracy 0.876 )


### 3) التقييم **بأيدينا** (كما في المعمل)
نحسب مصفوفة الالتباس ثمّ المقاييس بأنفسنا، دون أيّ مكتبة.

In [3]:
def confusion_matrix_by_hand(X, y_true, threshold):
    TP = FP = FN = TN = 0
    for i in range(len(X)):
        pred = predict(X[i], threshold)
        true = y_true[i]
        if pred == 1 and true == 1:   TP += 1
        elif pred == 1 and true == 0: FP += 1
        elif pred == 0 and true == 1: FN += 1
        else:                         TN += 1
    return TP, FP, FN, TN

TP, FP, FN, TN = confusion_matrix_by_hand(X_test, y_test, best_t)
acc  = (TP + TN) / (TP + FP + FN + TN)
prec = TP / (TP + FP)
rec  = TP / (TP + FN)
print("BY HAND ->  TP", TP, "FP", FP, "FN", FN, "TN", TN)
print("accuracy", round(acc, 3), "| precision", round(prec, 3), "| recall", round(rec, 3))

BY HAND ->  TP 42 FP 13 FN 8 TN 55
accuracy 0.822 | precision 0.764 | recall 0.84


### 4) التقييم **بـ scikit-learn**
نفس الأرقام، لكن بمكتبةٍ جاهزة. لاحظ ترتيب مصفوفة sklearn: صفوف = الحقيقة، أعمدة = التنبّؤ،
وترتيب الخانات عند التفكيك هو TN, FP, FN, TP.

In [4]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, classification_report

y_pred = [predict(x, best_t) for x in X_test]

cm = confusion_matrix(y_test, y_pred)          # [[TN, FP], [FN, TP]]
tn, fp, fn, tp = cm.ravel()
print("sklearn confusion matrix (TN, FP, FN, TP):", tn, fp, fn, tp)
print("precision:", round(precision_score(y_test, y_pred), 3))
print("recall   :", round(recall_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred, target_names=["not efficient", "efficient"]))

sklearn confusion matrix (TN, FP, FN, TP): 55 13 8 42
precision: 0.764
recall   : 0.84

               precision    recall  f1-score   support

not efficient       0.87      0.81      0.84        68
    efficient       0.76      0.84      0.80        50

     accuracy                           0.82       118
    macro avg       0.82      0.82      0.82       118
 weighted avg       0.83      0.82      0.82       118



### 5) قارِن: هل تطابق الحسابان؟
الدرس المهمّ: المكتبة لا تفعل سحرًا — إنّها تنفّذ نفس الخطوات التي كتبناها بأيدينا.

In [5]:
print("hand   :", TP, FP, FN, TN)
print("sklearn:", tp, fp, fn, tn)
print("match?", (TP, FP, FN, TN) == (tp, fp, fn, tn))

hand   : 42 13 8 55
sklearn: 42 13 8 55
match? True


### 6) الفخّ + المفاضلة
قارن بالنموذج الكسول، ثمّ جرّب عتبات وزنٍ مختلفة.

In [6]:
lazy_acc = sum(1 for t in y_test if t == 0) / len(y_test)
print("lazy 'not efficient' accuracy:", round(lazy_acc, 3), "(catches 0 efficient cars)\n")

print("weight_thr | precision recall")
for t in [2400, 2600, 2700, 2800, 3000]:
    yp = [predict(x, t) for x in X_test]
    print(f"   {t}    |   {precision_score(y_test, yp):.3f}   {recall_score(y_test, yp):.3f}")

lazy 'not efficient' accuracy: 0.576 (catches 0 efficient cars)

weight_thr | precision recall
   2400    |   0.892   0.660
   2600    |   0.760   0.760
   2700    |   0.764   0.840
   2800    |   0.770   0.940
   3000    |   0.671   0.980


### 7) قرارك وتبريرك  (اكتب إجابتك هنا)
1. **أيّ مقياسٍ يهمّ أكثر** لشارة «موفّرة للوقود» في تطبيق بيع: الضبط أم الاستدعاء؟ **ولماذا؟**
2. رفع عتبة الوزن يجعل النموذج يمنح الشارة لسياراتٍ أكثر. ماذا يحدث للضبط؟ وللاستدعاء؟
3. **أيّ عتبةٍ توصي بها** لإطلاق الشارة؟ اذكر ما تكسبه وما تخسره.
4. **فكرة معكوسة:** لو كان الهدف منح حافزٍ حكوميٍّ لكلّ سيارةٍ موفّرة (فلا نريد أن نفوّت أحدًا)،
   فأيّ مقياسٍ يصبح الأهمّ؟ وهل تتغيّر توصيتك؟

> اكتب إجابتك في خليةٍ نصّية.

### 8) تحدٍّ (اختياري)
استعمل `classification_report` لتقرأ الضبط والاستدعاء و F1 لكلٍّ من الفئتين معًا.
أيّ عتبةٍ تعطي أعلى F1 للفئة «الموفّرة»؟ وهل يتوافق ذلك مع قرارك في الخطوة السابقة؟